# Broadcast → .srt transcription (Kaggle T4 GPU)

Runs the `broadcast-srt` pipeline (faster-whisper `large-v3`, per-chunk language
detection) on a Darija / Arabic / French broadcast clip and verifies the output `.srt`.

**Setup:** Settings → Accelerator → **GPU T4 x2** (or P100). Internet **On** (first run
downloads the model weights).

## 1. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.0"

## 2. Get the pipeline code

Either clone the repo, or (as below) upload `src/transcribe.py` + `src/srt_writer.py`
as a Kaggle Dataset and point `SRC` at it. Adjust `SRC` to wherever the two files live.

In [ ]:
import sys, glob

# Point this at the folder containing transcribe.py + srt_writer.py.
SRC = "/kaggle/input/broadcast-srt/src"  # <-- edit to your dataset path
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import transcribe
from srt_writer import write_srt
print("loaded:", transcribe.__file__)

## 3. Pick a clip

Upload a short broadcast clip (mp3/mp4/wav/...) as a Kaggle Dataset and set `CLIP`.

In [ ]:
CLIP = "/kaggle/input/haca-sample/clip.mp4"  # <-- edit to your audio/video file
assert glob.glob(CLIP), f"clip not found: {CLIP}"
print("clip:", CLIP)

## 4. Load the model on GPU

`large-v3` in `float16`. First run downloads the weights (~3 GB).

In [ ]:
model = transcribe.load_model("large-v3", device="cuda", compute_type="float16")

## 5. Transcribe with per-chunk language detection

`lang="auto"` detects the language per ~25 s chunk (allow-list `ar,fr,en`), so a
code-switched broadcast transcribes natively.

In [ ]:
segments = transcribe.transcribe_file(
    CLIP, model,
    lang="auto", allowed=("ar", "fr", "en"),
    max_chunk_s=25.0, beam_size=5,
)
print(f"{len(segments)} cues")
for s in segments[:10]:
    print(f"[{s['start']:6.1f}-{s['end']:6.1f}] ({s['lang']}) {s['text']}")

# Per-language cue counts — sanity check that a mixed clip actually switched.
from collections import Counter
print("lang mix:", Counter(s["lang"] for s in segments))

## 6. Write the .srt

In [ ]:
out_path = write_srt(segments, "/kaggle/working/clip.srt")
print("wrote", out_path)
print(out_path.read_text(encoding="utf-8")[:1200])

## 7. Verify the output is a valid, parseable SRT

Re-parse with the same standard-SRT contract a downstream consumer uses (e.g. the HACA
benchmark's `srt_utils.parse_srt`). If this prints cues, the file is interoperable.

In [ ]:
import re
BLOCK = re.compile(r"(\d+)\n(\d\d:\d\d:\d\d,\d\d\d) --> (\d\d:\d\d:\d\d,\d\d\d)\n(.+?)(?=\n\n|\Z)", re.DOTALL)
text = out_path.read_text(encoding="utf-8")
cues = BLOCK.findall(text)
print(f"parsed {len(cues)} cues; first:", cues[0] if cues else None)
assert len(cues) == len(segments), "cue count mismatch — SRT format problem"